In [2]:
import sys
!{sys.executable} -m pip install python-dotenv openai qdrant-client sentence-transformers

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached qdrant_client-1.18.0-py3-none-any.whl.metadata (11 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp314-cp314-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached portalocker-3.2.0-py3-none-any.whl.metada


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from neo4j import GraphDatabase

# 1. Carrega as chaves e conecta nos bancos do seu Docker
load_dotenv()
client_maritaca = OpenAI(api_key=os.getenv("MARITACA_API_KEY"), base_url="https://chat.maritaca.ai/api")
qdrant_client = QdrantClient(url="http://localhost:6333")
encoder = SentenceTransformer("all-MiniLM-L6-v2")

neo4j_driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))

# =========================================================================
# ONDE VOCÊ COLOCA A PERGUNTA: O sistema vai abrir uma caixinha na tela!
# =========================================================================
pergunta_usuario = input("Digite a sua pergunta para o GraphRAG: ")
print(f"\n[1/5] Pergunta recebida: '{pergunta_usuario}'")

# --- PASSO 1: Maritaca expande a pergunta (Query Expansion) ---
prompt_expansao = f"Gere palavras-chave e sinônimos em inglês e português para a busca: {pergunta_usuario}"
resposta_exp = client_maritaca.chat.completions.create(
    model="sabiazinho-4",
    messages=[{"role": "user", "content": prompt_expansao}]
)
pergunta_expandida = resposta_exp.choices[0].message.content
print("[2/5] Maritaca expandiu os conceitos da pergunta...")

# --- PASSO 2: Busca Vetorial no Qdrant (Sintaxe Atualizada para v1.11+) ---
vetor_pergunta = encoder.encode(pergunta_expandida).tolist()

# Na versão nova, mudamos .search por .query_points, e query_vector por query
resposta_qdrant = qdrant_client.query_points(
    collection_name="imdb_top_1000_rag",  # Sua coleção validada no painel
    query=vetor_pergunta,                 # Parâmetro atualizado!
    limit=1
)

# Acessamos a lista de pontos retornados e extraímos o título do payload
resultados_points = resposta_qdrant.points
if len(resultados_points) > 0:
    filme_encontrado = resultados_points[0].payload["title"]
    print(f"[3/5] Qdrant localizou o filme mais próximo semanticamente: '{filme_encontrado}'")
else:
    raise ValueError("Nenhum filme correspondente foi encontrado na coleção do Qdrant.")

# --- PASSO 3: Travessia de Grafo no Neo4j ---
print("[4/5] Entrando no Neo4j para buscar os relacionamentos...")
contexto_grafo = ""
with neo4j_driver.session() as session:
    query_cypher = """
    MATCH (d:Diretor)-[:DIRIGIU]->(f:Filme {titulo: $titulo_filme})-[:PERTENCE_AO_GENERO]->(g:Genero)
    RETURN d.nome AS diretor, g.nome AS genero, f.resumo AS resumo
    """
    resultado_neo4j = session.run(query_cypher, titulo_filme=filme_encontrado)
    
    # Organiza os resultados encontrados no grafo
    diretores = set()
    generos = set()
    resumo = ""
    for registro in resultado_neo4j:
        diretores.add(registro['diretor'])
        generos.add(registro['genero'])
        resumo = registro['resumo']
        
    contexto_grafo = f"Filme: {filme_encontrado}\nDiretor(es): {', '.join(diretores)}\nGênero(s): {', '.join(generos)}\nSinopse: {resumo}"

# --- PASSO 4: Resposta Final da Maritaca com o Contexto Blindado ---
print("[5/5] Enviando dados amarrados para a Maritaca formular a resposta...")
prompt_sistema = (
    "Você é um recomendador de filmes inteligente. Use o contexto real fornecido "
    "(que veio do Neo4j e do Qdrant) para responder ao usuário. Seja simpático e fale em português."
)
prompt_usuario_final = f"Contexto do Banco de Dados:\n{contexto_grafo}\n\nPergunta do Usuário: {pergunta_usuario}"

resposta_final = client_maritaca.chat.completions.create(
    model="sabiazinho-4",
    messages=[
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": prompt_usuario_final}
    ]
)

print("\n" + "="*50)
print("=== RESPOSTA DO GRAPH RAG ===")
print(resposta_final.choices[0].message.content)
print("="*50)

neo4j_driver.close()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1523.43it/s]



[1/5] Pergunta recebida: 'Quem foi o diretor do filme Akira, ano e gênero?'
[2/5] Maritaca expandiu os conceitos da pergunta...
[3/5] Qdrant localizou o filme mais próximo semanticamente: 'Sanjuro'
[4/5] Entrando no Neo4j para buscar os relacionamentos...
[5/5] Enviando dados amarrados para a Maritaca formular a resposta...

=== RESPOSTA DO GRAPH RAG ===
Parece que houve uma pequena confusão no nome do filme que você mencionou. O filme sobre o qual tenho informações é "Sanjuro", dirigido por Akira Kurosawa. Ele pertence ao gênero de Ação (Action) e é um clássico do cinema japonês. 

Mas, se você estava se referindo ao filme "Akira" (1988), um famoso anime cyberpunk, o diretor foi Katsuhiro Otomo. O filme foi lançado em 1988 e pertence aos gêneros Animação, Ficção Científica e Ação.

Se quiser detalhes sobre "Sanjuro" ou curiosidades sobre "Akira", posso te contar mais!
